In [2]:
# model
from rich import print as rprint
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash", # 模型名称
)

In [3]:
from langchain_core.tools import tool

# Google 风格 Docstring
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取城市的天气信息

    Args:
        city : 具体的城市

    Returns:
        返回城市的天气信息
    """
    return city + "晴天，温度15°C"

## Middleware

### 内置中间件 (Prebuilt middleware)

https://docs.langchain.com/oss/python/langchain/middleware/built-in

### 自定义中间件 (Custom middleware)

#### 1、基于装饰器的实现

In [37]:
from typing import Callable, Any
from langchain.agents.middleware import before_agent, after_agent, before_model, after_model, wrap_model_call, \
    wrap_tool_call, ModelRequest, ModelResponse
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from langgraph.types import Command

# Node-style hooks
@before_agent
def log_before_agent(state: AgentState, runtime: Runtime) -> None:
    print(f"[before_agent]")
    # rprint(state)
    return None

@after_agent
def log_after_agent(state: AgentState, runtime: Runtime) -> None:
    print(f"[after_agent]")
    # rprint(state)
    return None

@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> None:
    print(f"[before_model]")
    # rprint(state)
    return None

@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> None:
    print(f"[after_model]")
    # rprint(state)
    return None

# Wrap-style hooks
@wrap_model_call                                      # Callable[[入参类型], 返回类型]
def log_wrap_model_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse | None:
    # rprint(request)

    print(f"before[wrap_model_call]")
    # 模型的调用
    model_response =  handler(request)
    print(f"after[wrap_model_call]")

    return model_response

@wrap_tool_call
def log_wrap_tool_call(request: ToolCallRequest, handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]]):
    # rprint(request)

    print(f"before[wrap_tool_call]")
    # 工具调用
    tool_response = handler(request)
    print(f"after[wrap_tool_call]")

    return tool_response


#### 2、基于类的实现

In [6]:
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from langgraph.types import Command
from typing import Callable, Any

class TestMiddleware(AgentMiddleware):
    def before_agent(self, state: AgentState, runtime: Runtime) -> None:
        print(f"[before_agent]")
        # rprint(state)
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> None:
        print(f"[after_agent]")
        # rprint(state)
        return None

    def before_model(self, state: AgentState, runtime: Runtime) -> None:
        print(f"[before_model]")
        # rprint(state)
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> None:
        print(f"[after_model]")
        # rprint(state)
        return None

    def wrap_model_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse | None:
        # rprint(request)

        print(f"before[wrap_model_call]")
        # 模型的调用
        model_response =  handler(request)
        print(f"after[wrap_model_call]")

        return model_response

    def wrap_tool_call(self, request: ToolCallRequest, handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]]) -> ToolMessage | Command[Any]:
        # rprint(request)

        print(f"before[wrap_tool_call]")
        # 工具的调用
        tool_response = handler(request)
        print(f"after[wrap_tool_call]")

        return tool_response

In [7]:
 # agent
from langchain.agents import create_agent

agent = create_agent(
    name="test_agent",
    model = model,
    system_prompt="你是一个话少的助手",
    tools=[get_weather],
    # middleware=[log_before_agent, log_after_agent, log_before_model, log_after_model, log_wrap_model_call, log_wrap_tool_call]
    middleware=[TestMiddleware()]
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "苏州的天气如何"},
    ]
})

rprint(response)

[before_agent]
[before_model]
before[wrap_model_call]
after[wrap_model_call]
[after_model]
before[wrap_tool_call]
after[wrap_tool_call]
[before_model]
before[wrap_model_call]
after[wrap_model_call]
[after_model]
[after_agent]


{
    'messages': [
        HumanMessage(
            content='苏州的天气如何',
            additional_kwargs={},
            response_metadata={},
            id='22cc77c4-743c-4bb1-8e32-274dfe5259f1'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '用户想知道苏州的天气。我需要调用get_weather工具来获取苏州的天气信息。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 64,
                    'prompt_tokens': 288,
                    'total_tokens': 352,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 19,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 32
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': 'b9d4546d-537f-4484-98e5-7c1a90234c03',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='test_agent',
            id='lc_run--019f9d57-673f-7952-8459-34bd2cbaa0e6-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '苏州'},
                    'id': 'call_00_Z3SwlgcY0kwjcN7K3W5x9443',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 288,
                'output_tokens': 64,
                'total_tokens': 352,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {'reasoning': 19}
            }
        ),
        ToolMessage(
            content='苏州晴天，温度15°C',
            name='get_weather',
            id='955f3324-07cc-4870-83ff-7b99d7abf612',
            tool_call_id='call_00_Z3SwlgcY0kwjcN7K3W5x9443'
        ),
        AIMessage(
            content='苏州晴天，15°C。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '好的，苏州的天气是晴天，温度15°C。我直接简洁地回答用户。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 27,
                    'prompt_tokens': 371,
                    'total_tokens': 398,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 19,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 115
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '72da184f-92b3-42a5-aab0-b9c455170104',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='test_agent',
            id='lc_run--019f9d57-6e4c-7852-9078-978e76d3b201-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 371,
                'output_tokens': 27,
                'total_tokens': 398,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {'reasoning': 19}
            }
      